# Reverse carry, breakeven check

Reverse carry is long perp short spot, to collect funding when it is negative and shorts pay longs. Collecting it means holding a short spot position, which costs a borrow fee every day. So it only works when the funding collected beats the borrow cost.

We do not have historical borrow rates. So instead of guessing them, this notebook solves for the breakeven borrow rate, the daily spot borrow cost that exactly cancels the funding collected. Then the decision is one comparison, is a real borrow rate below the breakeven.

## What this is and is not

- Data is the futures funding history we already have, nothing bought.
- This is a viability filter, deliberately generous. It assumes you are in the position only during negative-funding prints and ignores borrow paid on the positive prints you would hold through.
- Availability is not modelled. Whether you could actually borrow a name in a given period is a live-only fact, testnet answers it. The biggest edges tend to be the least borrowable.
- Fee drag depends on a hold-length assumption, so borderline names move with it.

In [1]:
import os, glob
import pandas as pd, numpy as np

DATA = os.path.expanduser("~/quant-platform/data/binance_historical")
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
           "DOGEUSDT", "ADAUSDT", "LINKUSDT", "AVAXUSDT", "LTCUSDT"]

# Assumptions. Change these and the borderline names move.
FEE_ROUND_TRIP_BPS = 34.0   # both legs, taker, per round trip
HOLD_DAYS          = 10      # for amortising the round-trip fee
ASSUMED_BORROW     = 5.0     # bps per day, a plausible major-coin borrow rate

# funding csv has no header: symbol, kind, ts_ms, interval, rate
COLS = ["symbol", "kind", "ts_ms", "interval", "rate"]


def load_funding(sym):
    fs = sorted(glob.glob(f"{DATA}/{sym}/futures/funding/*.csv"))
    d = pd.concat([pd.read_csv(f, header=None, names=COLS) for f in fs], ignore_index=True)
    d["ts"] = pd.to_datetime(d["ts_ms"], unit="ms")
    d["rate"] = pd.to_numeric(d["rate"], errors="coerce")
    return d[["ts", "rate"]]


funding = {s: load_funding(s) for s in SYMBOLS}
print("loaded", {s: len(d) for s, d in funding.items()})

loaded {'BTCUSDT': 3288, 'ETHUSDT': 3288, 'SOLUSDT': 3363, 'BNBUSDT': 3288, 'XRPUSDT': 3288, 'DOGEUSDT': 3288, 'ADAUSDT': 3288, 'LINKUSDT': 3288, 'AVAXUSDT': 3288, 'LTCUSDT': 3288}


In [2]:
rows = []
for sym in SYMBOLS:
    d = funding[sym].copy()
    d["yr"] = d["ts"].dt.year
    for yr, g in d.groupby("yr"):
        if yr < 2022 or yr > 2024:
            continue
        f = g["rate"]
        neg = f[f < 0]
        n, nn = len(f), len(neg)
        mean_neg_bps = neg.mean() * 1e4 if nn else 0.0        # funding on negative prints
        collected_day = abs(mean_neg_bps) * 3                  # 3 prints per day, bps per day
        total_neg_yr = -neg.sum() * 1e4 if nn else 0.0         # whole-year opportunity, bps
        fee_drag_day = FEE_ROUND_TRIP_BPS / HOLD_DAYS
        breakeven = collected_day - fee_drag_day               # max borrow you can pay
        rows.append({"symbol": sym, "year": int(yr), "frac_neg": round(nn / n, 3),
                     "mean_neg_bps": round(mean_neg_bps, 2),
                     "collected_bpsday": round(collected_day, 2),
                     "total_neg_bps_yr": round(total_neg_yr, 0),
                     "breakeven_borrow_bpsday": round(breakeven, 2),
                     "net_vs_assumed": round(breakeven - ASSUMED_BORROW, 2)})

table = pd.DataFrame(rows)
# viable where the breakeven borrow clears the assumed borrow
table["viable"] = table["net_vs_assumed"] > 0
display(table.sort_values(["symbol", "year"]).reset_index(drop=True))
print("\nviable symbol-years at", ASSUMED_BORROW, "bps/day borrow:")
print(table[table["viable"]][["symbol", "year", "breakeven_borrow_bpsday"]].to_string(index=False))

,symbol,year,frac_neg,mean_neg_bps,collected_bpsday,total_neg_bps_yr,breakeven_borrow_bpsday,net_vs_assumed,viable
0,ADAUSDT,2022,0.409,-1.33,3.99,596.0,0.59,-4.41,False
1,ADAUSDT,2023,0.174,-1.00,3.00,190.0,-0.40,-5.40,False
2,ADAUSDT,2024,0.084,-0.43,1.28,39.0,-2.12,-7.12,False
3,AVAXUSDT,2022,0.479,-1.85,5.55,971.0,2.15,-2.85,False
4,AVAXUSDT,2023,0.181,-1.20,3.59,237.0,0.19,-4.81,False
5,AVAXUSDT,2024,0.230,-1.06,3.18,268.0,-0.22,-5.22,False
6,BNBUSDT,2022,0.506,-2.28,6.85,1265.0,3.45,-1.55,False
7,BNBUSDT,2023,0.294,-3.73,11.19,1201.0,7.79,2.79,True
8,BNBUSDT,2024,0.221,-3.67,11.02,892.0,7.62,2.62,True
9,BTCUSDT,2022,0.221,-0.64,1.91,154.0,-1.49,-6.49,False



viable symbol-years at 5.0 bps/day borrow:
 symbol  year  breakeven_borrow_bpsday
SOLUSDT  2022                    20.57
BNBUSDT  2023                     7.79
BNBUSDT  2024                     7.62


Mostly dead. For nearly every symbol and year the negative funding is too small to cover even the round-trip fee, before borrow enters, so the breakeven borrow is negative. Reverse carry is not a broad strategy.

Two pockets stand out.

SOL 2022 is enormous, 24 bps a day while negative, 4283 bps over the year, breakeven borrow 20 bps a day. But that is the FTX crash, one name one year, and SOL was exactly the thing you could not borrow in late 2022. The biggest edge is where the trade was impossible. Treat it as a warning, not an opportunity.

BNB is the real one. Persistent negative funding every year, frac negative 0.5 then 0.29 then 0.22, breakeven borrow 3.5 to 7.8 bps a day, and it clears a 5 bps borrow in 2023 and 2024. BNB is a major exchange token, easy to borrow. Structural negative funding plus borrowable is the combination we want. If reverse carry is anything here, it is a BNB trade.

Caveat, the marginal names swing on the hold-length assumption, 34 bps over 10 days is 3.4 bps a day of fee drag. Hold longer and more of them clear, so the borderline cases are assumption dependent. SOL 2022 and BNB are not, they clear comfortably.

Conclusion, not worth the historical sim work. It is a BNB bear-regime trade, funding negative only a fifth to a half of the time and shrinking, and the big numbers are unborrowable distress like SOL 2022. Rather than build a short-spot and borrow-cost mechanic to backtest it, add the symmetric reverse leg to the live strategy and judge it on testnet, where the borrow rate and availability are real and free.